[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/reto_expres/Reto_expres_5_semanas_raras.ipynb)

# Reto exprés S12-13 · Autoencoder: ¿qué semanas hay que revisar a mano?

**Solo se tocan las celdas ✏️** (cambiar un valor y pulsar ▶). Las ⚙️ se ejecutan tal cual.

- ✏️ **1** · probar 3 tamaños del «cuello» del autoencoder y compararlo con una regla simple, **mirando solo la validación**
- ✏️ **2** · elegir el **umbral** según lo que cuesta no ver una semana rara y lo que cuesta revisar una normal
- ✏️ **3** · contadlo: titular con cifra y gráfico
- ✏️ **4** · la aplicación: revisar una semana
- Al final sale una **FICHA** con todos los números para vuestra presentación.

| Qué se hace | Datos | Rival simple | Métrica |
|---|---|---|---|
| marcar las semanas de ocupación que no se parecen a las normales | 156 semanas de `ocupacion_diaria.csv` (con 10 semanas raras **inyectadas** en validación y prueba) | regla «la media de la semana se sale de ±2 desviaciones» | AUC del error de reconstrucción (más alto = mejor) y coste |

### ⚙️ Preparación — pulsa ▶

In [ ]:
#@title ⚙️ Ejecuta esta celda primero
import os, re, urllib.request, time
BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"
def dato(nombre):
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta): return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEMILLA = 42

from tensorflow import keras
keras.utils.set_random_seed(SEMILLA)

from sklearn.metrics import roc_auc_score
print("Listo ✔")

### ⚙️ Preparar las semanas — pulsa ▶

- **Entrenamiento**: las primeras 104 semanas (el autoencoder solo ve semanas **normales**).
- **Validación** y **prueba**: 26 semanas cada una, con **5 semanas raras inyectadas** en cada una (se sabe cuáles): unas con **el patrón de la semana al revés** (llena entre semana y vacía el fin de semana, **con la misma media**) y otras con **3 días hundidos 22 puntos** (obras, cierre).
- Así podemos medir si las encuentra. En la vida real nadie te dice cuáles son.

In [ ]:
d = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"])
Y = d.ocupacion_pct.to_numpy(float)[:1092].reshape(156, 7).copy()          # 156 semanas de 7 días
inicio = d.fecha.to_numpy()[:1092:7]
mu, sd = Y[:104].mean(), Y[:104].std()
rng = np.random.default_rng(SEMILLA); rara = np.zeros(156, int)
for tramo in (range(104, 130), range(130, 156)):
    for orden, w in enumerate(sorted(rng.choice(list(tramo), 5, replace=False))):
        rara[w] = 1
        if orden % 2 == 0: m = Y[w].mean(); Y[w] = m + 0.7 * (m - Y[w])      # el patrón de la semana, al revés (sin cambiar la media)
        else: Y[w, 2:5] = Y[w, 2:5] - 22                                    # tres días hundidos
Z = ((Y - mu) / sd).astype("float32")
Z_ent, Z_val, Z_pru = Z[:104], Z[104:130], Z[130:]; r_val, r_pru = rara[104:130], rara[130:]
error = lambda m, A: ((A - m.predict(A, verbose=0)) ** 2).mean(1)
medias_ent = Z_ent.mean(1)
regla = lambda A: np.abs(A.mean(1) - medias_ent.mean()) / medias_ent.std()   # rival: ¿la media de la semana se sale de lo normal?
print(f"{len(Z_ent)} semanas de entrenamiento · validación {len(Z_val)} ({r_val.sum()} raras) · prueba {len(Z_pru)} ({r_pru.sum()} raras)")
print(f"Rival «la media se sale de ±2 desviaciones»: AUC en validación {roc_auc_score(r_val, regla(Z_val)):.3f}")

### ⚙️ El autoencoder — pulsa ▶

Comprime 7 números (una semana) a **N** (el cuello) y los reconstruye. Si una semana se reconstruye **mal**, no se parece a las normales.

In [ ]:
def entrenar(cuello, epocas=300):
    keras.utils.set_random_seed(SEMILLA)
    m = keras.Sequential([keras.layers.Input((7,)), keras.layers.Dense(cuello, activation="tanh"), keras.layers.Dense(7)])
    m.compile("adam", "mse"); m.fit(Z_ent, Z_ent, epochs=epocas, batch_size=16, verbose=0); return m
print("Listo ✔")

### ✏️ 1 · Probar tamaños del cuello y elegir (solo con la validación)

Cambiad **CUELLOS** (mínimo 3 valores, entre 1 y 6). Mirad **val_auc** (1 = encuentra todas las raras; 0,5 = azar) y comparadlo con la regla simple.

In [ ]:
CUELLOS = [2, 3, 5]
filas = [dict(candidata="regla: la media se sale de lo normal (rival)", val_auc=round(float(roc_auc_score(r_val, regla(Z_val))), 3), segundos="-")]
for k in CUELLOS:
    t0 = time.time(); m = entrenar(k)
    filas.append(dict(candidata=f"autoencoder con cuello {k}", val_auc=round(float(roc_auc_score(r_val, error(m, Z_val))), 3), segundos=round(time.time() - t0)))
tabla = pd.DataFrame(filas); display(tabla)
ELEGIDA = 3                # ✏️ cuello elegido, justificado con la tabla de arriba

### ✏️ 2 · El umbral: ¿qué cuesta más? (solo con la validación)

El umbral es el **percentil** del error de entrenamiento a partir del cual se marca una semana. Más bajo = se marcan más semanas (se escapan menos raras, pero hay que revisar más normales).

- **COSTE_ESCAPA**: una semana rara que nadie ve (€ de una mala decisión).
- **COSTE_REVISAR**: revisar a mano una semana normal (€ de tiempo).

In [ ]:
COSTE_ESCAPA = 100       # ✏️ € por semana rara que se escapa
COSTE_REVISAR = 10       # ✏️ € por revisar una semana normal
modelo = entrenar(ELEGIDA); e_ent, e_val = error(modelo, Z_ent), error(modelo, Z_val)
def evaluar_umbral(pc, e, r):
    u = np.percentile(e_ent, pc); marca = e > u
    tp, fp, fn = int((marca & (r == 1)).sum()), int((marca & (r == 0)).sum()), int((~marca & (r == 1)).sum())
    return u, tp, fp, fn, fn * COSTE_ESCAPA + fp * COSTE_REVISAR
filas_u = []
for pc in [80, 90, 95, 99]:
    u, tp, fp, fn, cs = evaluar_umbral(pc, e_val, r_val); filas_u.append(dict(percentil=pc, umbral=round(float(u), 3), detecta=tp, falsas_alarmas=fp, se_escapan=fn, coste_eur=cs))
display(pd.DataFrame(filas_u)); print("Sin vigilar nada: se escaparían", int(r_val.sum()), "semanas raras =", int(r_val.sum()) * COSTE_ESCAPA, "€")
PERCENTIL = 95           # ✏️ percentil elegido

### ⚙️ La prueba, una sola vez — pulsa ▶

In [ ]:
e_pru = error(modelo, Z_pru); u, tp, fp, fn, cs = evaluar_umbral(PERCENTIL, e_pru, r_pru); tn = int((r_pru == 0).sum()) - fp
auc_ae, auc_regla = roc_auc_score(r_pru, e_pru), roc_auc_score(r_pru, regla(Z_pru)); sin_vigilar = int(r_pru.sum()) * COSTE_ESCAPA
FICHA = dict(elegida=ELEGIDA, percentil=PERCENTIL, auc_regla=round(auc_regla, 3), auc=round(auc_ae, 3), detecta=tp, raras=int(r_pru.sum()), falsas_alarmas=fp, se_escapan=fn,
             coste=cs, sin_vigilar=sin_vigilar)
print("Semanas que se escapan:", [str(inicio[130 + i])[:10] for i in np.where((e_pru <= u) & (r_pru == 1))[0]])
print("Falsas alarmas (semanas normales marcadas):", [str(inicio[130 + i])[:10] for i in np.where((e_pru > u) & (r_pru == 0))[0]])
print(f"""
================ FICHA PARA LA PRESENTACIÓN ================
Problema ........ marcar semanas de ocupación raras · {len(Z_pru)} semanas de prueba ({int(r_pru.sum())} raras)
Candidatas ...... {[(f['candidata'], f['val_auc']) for f in filas]}  (AUC en validación)
Elegida ......... autoencoder con cuello {ELEGIDA} · percentil {PERCENTIL}   (costes: escapa {COSTE_ESCAPA} €, revisar {COSTE_REVISAR} €)
--- PRUEBA (una sola vez) ---
AUC regla simple . {auc_regla:.3f}      AUC autoencoder . {auc_ae:.3f}   {"(gana el autoencoder)" if auc_ae > auc_regla + 0.02 else "(NO gana de forma clara)"}
Detecta {tp} de {int(r_pru.sum())} raras · {fp} falsas alarmas · se escapan {fn}
Coste {cs} € frente a {sin_vigilar} € sin vigilar
============================================================
""")

### ✏️ 3 · Contadlo (S13): titular con cifra y gráfico

Cambiad **TITULAR**. Cada barra es una semana de la prueba; la línea es el **umbral**; en rojo, las raras de verdad.
**Cómo se cuenta (S13):** el titular es una **conclusión con cifra** (no «Resultados»), el gráfico empieza en **0** y las cifras salen de variables (f-string), no se teclean a mano. Terminad con un **límite**: ¿cuándo no os fiaríais?

In [ ]:
TITULAR = f"Detectamos {tp} de {int(r_pru.sum())} semanas raras y revisamos {fp} normales: {sin_vigilar - cs} € menos que sin vigilar"   # ✏️ vuestra conclusión
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(range(len(e_pru)), e_pru, color=["tab:red" if x else "tab:blue" for x in r_pru]); ax.axhline(u, color="black", ls="--", label=f"umbral (percentil {PERCENTIL})")
ax.set_xticks(range(len(e_pru)), [str(inicio[130 + i])[5:10] for i in range(len(e_pru))], rotation=70, fontsize=7)
ax.set_ylabel("error de reconstrucción"); ax.set_ylim(0, None); ax.legend(); ax.set_title("Rojo = semana rara inyectada · azul = semana normal")
fig.suptitle(TITULAR, fontsize=11, fontweight="bold"); plt.tight_layout(); plt.show()
FICHA["titular"] = TITULAR

### ✏️ 4 · La aplicación: revisar una semana

`revisar([...])` recibe **7 ocupaciones** (lunes a domingo) y dice si hay que revisarla. Probad una semana normal, una con el patrón al revés y una inventada por vosotros.

In [ ]:
def revisar(semana):
    semana = [round(float(v), 1) for v in semana]
    A = ((np.array(semana, dtype="float32") - mu) / sd).reshape(1, 7); e = float(error(modelo, A)[0])
    print(f"{semana} → error {e:.3f} (umbral {u:.3f}): " + ("REVISAR A MANO" if e > u else "normal"))

revisar(list(Y[130])); revisar([88, 85, 80, 70, 62, 60, 58]); revisar([60, 60, 60, 60, 60, 60, 60])

### 🎤 Para la presentación (2–3 min)

Copiad la **FICHA**, la tabla de la celda ✏️ 1 y el gráfico. Una frase para cada pregunta: **¿gana a la regla simple? · ¿por qué ese umbral? · ¿qué semanas se escapan o se marcan de más? · ¿quién revisa lo marcado?**